## 5. Association Rules Mining

## Section 1

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import joblib
import os

## Section 3

In [2]:

df = pd.read_csv('data/academic_dataset.csv')

print(f'حجم البيانات (عدد الصفوف، عدد الأعمدة): {df.shape}') 

print(f'\nأسماء الأعمدة المتوفرة: {list(df.columns)}') 

print(f'\nأول 5 صفوف من البيانات للتوضيح:') 
df.head()

## 3. Exploratory Data Analysis (EDA)

In [3]:

print("=== معلومات تقنية حول الأعمدة ونوع البيانات ===")
df.info()

print("\n=== الملخص الإحصائي الشامل للبيانات ===")
print(df.describe(include="all"))

def count_courses(value):
    if pd.isna(value) or str(value).strip() == "":
        return 0
    courses_list = str(value).split("|")
    return len(courses_list)

df["num_passed_courses"] = df["Courses_Passed"].apply(count_courses)

df["num_failed_courses"] = df["Courses_Failed"].apply(count_courses)


target_encoder = LabelEncoder()
df["Track"] = target_encoder.fit_transform(df["Track"].astype(str))

print("\nشكل البيانات بعد المعالجة الأولية وإضافة أعمدة عدد المواد (النجاح والرسوب):")
print(df.head())

In [4]:
df.head(50)

In [5]:
print('=== توزيع الطلاب على المسارات الأكاديمية (العمود المستهدف) ===')
print(df['Track'].value_counts())

print(f'\n=== عدد القيم المفقودة (Nulls) في كل عمود ===')
print(df.isnull().sum())

In [6]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='Track', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('توزيع الطلاب على المسارات الأكاديمية (Track Distribution)')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='GPA', kde=True, ax=axes[0, 1], color='blue')
axes[0, 1].set_title('توزيع المعدل التراكمي للطلاب (GPA Distribution)')

sns.boxplot(data=df, x='Track', y='GPA', ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('توزيع المعدل التراكمي حسب المسار الأكاديمي (GPA by Track)')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.scatterplot(data=df, x='Attendance_Rate', y='GPA', hue='Track', ax=axes[1, 1], palette='viridis', alpha=0.6)
axes[1, 1].set_title('نسبة الحضور مقابل المعدل التراكمي (Attendance vs GPA)')

plt.tight_layout()

plt.savefig('data/eda_plots.png', dpi=150, bbox_inches='tight')

plt.show()
print('تم رسم وحفظ الرسومات التوضيحية الأربعة لبيانات الطلاب بنجاح!')

In [7]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='Study_Hours_Per_Week', kde=True, ax=axes[0], color='green')
axes[0].set_title('توزيع ساعات الدراسة الأسبوعية للطلاب (Study Hours Distribution)')

sns.boxplot(data=df, x='Track', y='Study_Hours_Per_Week', ax=axes[1], palette='viridis')
axes[1].set_title('ساعات الدراسة الأسبوعية لكل مسار (Study Hours by Track)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [8]:

all_courses = ['CS101', 'CS102', 'CS201', 'CS202', 'CS210', 'CS220', 'CS301', 'CS302', 'CS310', 'CS320',
               'CS330', 'CS340', 'CS350', 'CS360', 'CS401', 'CS410', 'CS420', 'CS430', 'CS440', 'CS450',
               'MATH101', 'MATH102', 'MATH201', 'MATH202', 'MATH301', 'MATH302', 'MATH310', 'MATH320',
               'ENG101', 'ENG102', 'ENG201', 'ENG210', 'ENG220', 'ENG301', 'ENG310', 'ENG320']

for course in all_courses:
    df[f'Passed_{course}'] = df['Courses_Passed'].apply(lambda val: 1 if course in str(val) else 0)
    
    df[f'Failed_{course}'] = df['Courses_Failed'].apply(lambda val: 1 if course in str(val) else 0)

print(f'تم بنجاح تحويل {len(all_courses)} مادة إلى {len(all_courses)*2} عمود ثنائي (Passed و Failed)')
print(f'الحجم الجديد لمصفوفة البيانات (الصفوف، الأعمدة): {df.shape}')

In [9]:

passed_cols = [f'Passed_{c}' for c in all_courses]
df['Total_Passed'] = df[passed_cols].sum(axis=1)

failed_cols = [f'Failed_{c}' for c in all_courses]
df['Total_Failed'] = df[failed_cols].sum(axis=1)

df['Pass_Fail_Ratio'] = df['Total_Passed'] / (df['Total_Failed'] + 1)

df['GPA_Credit_Ratio'] = df['GPA'] / (df['Credit_Hours_Completed'] / 100.0)

cs_passed_cols = [f'Passed_{c}' for c in all_courses if c.startswith('CS')]
df['CS_Passed'] = df[cs_passed_cols].sum(axis=1)

math_passed_cols = [f'Passed_{c}' for c in all_courses if c.startswith('MATH')]
df['Math_Passed'] = df[math_passed_cols].sum(axis=1)

eng_passed_cols = [f'Passed_{c}' for c in all_courses if c.startswith('ENG')]
df['Eng_Passed'] = df[eng_passed_cols].sum(axis=1)

cs_failed_cols = [f'Failed_{c}' for c in all_courses if c.startswith('CS')]
df['CS_Failed'] = df[cs_failed_cols].sum(axis=1)

math_failed_cols = [f'Failed_{c}' for c in all_courses if c.startswith('MATH')]
df['Math_Failed'] = df[math_failed_cols].sum(axis=1)

eng_failed_cols = [f'Failed_{c}' for c in all_courses if c.startswith('ENG')]
df['Eng_Failed'] = df[eng_failed_cols].sum(axis=1)

print('تم بنجاح حساب جميع السمات والمؤشرات الأكاديمية التخصصية الجديدة!')
print(df[['Total_Passed', 'Total_Failed', 'Pass_Fail_Ratio', 'GPA_Credit_Ratio', 'CS_Passed', 'Math_Passed', 'Eng_Passed']].head())

In [10]:

interest_mapping = {
    'machine learning': 0, 'deep learning': 0, 'neural networks': 0, 'computer vision': 0, 'NLP': 0, 'robotics': 0,
    'web development': 1, 'mobile apps': 1, 'software design': 1, 'agile': 1, 'testing': 1, 'DevOps': 1,
    'data analysis': 2, 'statistics': 2, 'visualization': 2, 'big data': 2, 'business intelligence': 2, 'ML': 2,
    'networking': 3, 'routing': 3, 'protocols': 3, 'telecommunications': 3, 'IoT': 3, 'wireless': 3,
    'security': 4, 'cryptography': 4, 'ethical hacking': 4, 'penetration testing': 4, 'forensics': 4, 'firewalls': 4,
    'cloud': 5, 'AWS': 5, 'Azure': 5, 'virtualization': 5, 'containers': 5, 'microservices': 5
}

df['Interest_Encoded'] = df['Interest_Area'].astype(str).map(interest_mapping)

print('تم تحويل الاهتمامات النصية للطالب إلى فئات رقمية مشفرة بنجاح!')
print(df[['Interest_Area', 'Interest_Encoded']].head(10))

In [11]:
df.head(50)

## 5. Association Rules Mining

In [12]:

transactions = []

for idx, row in df.iterrows():
    student_transaction = []
    
    for course in all_courses:
        if row[f'Passed_{course}'] == 1:
            student_transaction.append(course)
            
    student_transaction.append(f'Interest_{row["Interest_Encoded"]}')
    
    if row['GPA'] >= 3.5:
        student_transaction.append('High_GPA')
    elif row['GPA'] >= 2.5:
        student_transaction.append('Medium_GPA')
    else:
        student_transaction.append('Low_GPA')
        
    student_transaction.append(f'Track_{row["Track"]}')
    
    transactions.append(student_transaction)

print(f'إجمالي عدد المعاملات (المطابق تماماً لعدد الصفوف): {len(transactions)}')
print(f'عينة للمواد والصفات في سلة الطالب الأول: {transactions[0][:7]}...')

In [13]:

te = TransactionEncoder()

te_ary = te.fit(transactions).transform(transactions) 

df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

print(f'أبعاد مصفوفة المعاملات الثنائية (الصفوف، السمات): {df_transactions.shape}')
print(f'أول 10 أعمدة تم إنشاؤها في المصفوفة: {list(df_transactions.columns)[:10]}...')

In [14]:

frequent_itemsets = apriori(df_transactions, min_support=0.1, use_colnames=True, max_len=4)

frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f'تم بنجاح استخراج {len(frequent_itemsets)} مجموعة سلوك ومواد دراسية متكررة!')
print(f'\nأعلى 15 مجموعة تكراراً بين طلاب الكلية:')
print(frequent_itemsets.head(15))

In [15]:

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)

rules = rules.sort_values('confidence', ascending=False)

print(f'تم استخراج {len(rules)} قاعدة ارتباط أكاديمية من البيانات!')
print(f'\nأعلى 20 قاعدة ارتباط ثقة في الإرشاد الأكاديمي:')
top_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)
print(top_rules.to_string())

In [16]:

track_rules = rules[rules['consequents'].apply(lambda x: any('Track_' in str(item) for item in x))]

track_rules = track_rules.sort_values('confidence', ascending=False)

print(f'\n=== أفضل 15 قاعدة ارتباط موثوقة لتوجيه الطلاب نحو المسار الأكاديمي المناسب ===')
for idx, row in track_rules.head(15).iterrows():
    antecedents_str = ', '.join([str(item) for item in row['antecedents']])
    consequents_str = ', '.join([str(item) for item in row['consequents']])
    
    print(f'إذا كان سجل الطالب يحتوي على: {{{antecedents_str}}}')
    print(f'👈 فإنه يُنصح بالتوجه نحو المسار الدراسي: {{{consequents_str}}}')
    print(f'📊 [نسبة الثقة: {row["confidence"]:.1%} | قوة الدعم: {row["support"]:.1%} | معامل الرفع: {row["lift"]:.2f}]')
    print('-' * 80)

In [17]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(rules['confidence'], kde=True, ax=axes[0], color='purple')
axes[0].set_title('توزيع مستويات الثقة في القواعد (Rule Confidence)')
axes[0].set_xlabel('نسبة الثقة (Confidence)')

sns.scatterplot(data=rules.head(50), x='support', y='confidence', hue='lift', ax=axes[1], palette='viridis', s=100)
axes[1].set_title('الدعم مقابل الثقة وتأثير الرفع لأفضل 50 قاعدة (Support vs Confidence)')

plt.tight_layout()
plt.savefig('data/association_rules_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('تم حفظ الرسوم البيانية الخاصة بقواعد الارتباط في مجلد data بنجاح!')

## Classification Models (Track Prediction)

In [18]:


course_features = [f'Passed_{c}' for c in all_courses] + [f'Failed_{c}' for c in all_courses]

engineered_features = ['Total_Passed', 'Total_Failed', 'Pass_Fail_Ratio', 'GPA_Credit_Ratio',
                       'CS_Passed', 'Math_Passed', 'Eng_Passed', 'CS_Failed', 'Math_Failed', 'Eng_Failed']

numerical_features = ['GPA', 'Credit_Hours_Completed', 'Attendance_Rate', 'Study_Hours_Per_Week', 'Previous_Semester_GPA']

categorical_features = ['Interest_Encoded']

feature_columns = course_features + engineered_features + numerical_features + categorical_features

X = df[feature_columns]

y_track = df['Track']

print(f'عدد السمات/المدخلات الكلية المستخدمة للتنبؤ: {len(feature_columns)} ميزة')
print(f'حجم مصفوفة المدخلات X: {X.shape} | حجم العمود المستهدف y: {y_track.shape}')

In [19]:

le = LabelEncoder() 
y_track_encoded = le.fit_transform(y_track) 

print(f'التخصصات/المسارات الأصلية: {le.classes_}')
print(f'خريطة التشفير الرقمي: {dict(zip(le.classes_, range(len(le.classes_))))}')

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_track_encoded, 
    test_size=0.2,
    random_state=42,
    stratify=y_track_encoded
)

print(f'\nحجم مجموعة التدريب للتصنيف: {X_train_cls.shape[0]} طالب')
print(f'حجم مجموعة الاختبار للتصنيف: {X_test_cls.shape[0]} طالب')

scaler_cls = StandardScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls) 
X_test_cls_scaled = scaler_cls.transform(X_test_cls) 

print('تم تقسيم البيانات وتقييسها بالتساوي بنجاح!')

## Regression Models (GPA Prediction)

In [20]:

regression_features = course_features + engineered_features + [
    'Credit_Hours_Completed', 'Attendance_Rate', 'Study_Hours_Per_Week', 'Previous_Semester_GPA', 'Interest_Encoded'
]

X_reg = df[regression_features]

y_gpa = df['GPA']

print(f'عدد السمات لمدخلات الانحدار: {len(regression_features)} ميزة')
print(f'أبعاد المدخلات X_reg: {X_reg.shape} | أبعاد المستهدف y_gpa: {y_gpa.shape}')
print(f'\nإحصاءات المعدل التراكمي (الهدف المراد التنبؤ به):')
print(y_gpa.describe())

In [21]:

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_gpa, 
    test_size=0.2, 
    random_state=42
)

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print(f'حجم مجموعة تدريب الانحدار: {X_train_reg.shape[0]} عينة طالب')
print(f'حجم مجموعة اختبار الانحدار: {X_test_reg.shape[0]} عينة طالب')
print('تم تقسيم وتقييس بيانات توقع المعدل الدراسي بنجاح!')

## Classification Models (Track Prediction)

In [22]:

cls_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

cls_results = {}

for name, model in cls_models.items():
    model.fit(X_train_cls_scaled, y_train_cls)
    
    y_pred = model.predict(X_test_cls_scaled)
    
    acc = accuracy_score(y_test_cls, y_pred)
    
    f1 = f1_score(y_test_cls, y_pred, average='weighted')
    
    cv_scores = cross_val_score(model, X_train_cls_scaled, y_train_cls, cv=5, scoring='accuracy')
    
    cls_results[name] = {
        'accuracy': acc,
        'f1_score': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred
    }
    
    print(f'\n=== نتائج نموذج: {name} (تصنيف) ===')
    print(f'🎯 دقة التنبؤ (Accuracy): {acc:.2%}')
    print(f'📈 مقياس جودة التصنيف F1-Score: {f1:.4f}')
    print(f'🔄 متوسط التحقق المتقاطع (CV Accuracy): {cv_scores.mean():.2%} (+/- {cv_scores.std()*2:.4f})')

print('\n=== جدول مقارنة نماذج التصنيف الأولية ===')
for name, res in cls_results.items():
    print(f'📌 {name} -> الدقة: {res["accuracy"]:.2%}, مقياس F1: {res["f1_score"]:.4f}, ثبات النموذج: {res["cv_mean"]:.2%}')

## Regression Models (GPA Prediction)

In [23]:

reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(random_state=42, n_estimators=100)
}

reg_results = {}

for name, model in reg_models.items():
    model.fit(X_train_reg_scaled, y_train_reg)
    
    y_pred_reg = model.predict(X_test_reg_scaled)
    
    mse = mean_squared_error(y_test_reg, y_pred_reg)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_reg, y_pred_reg)
    r2 = r2_score(y_test_reg, y_pred_reg)
    
    cv_scores = cross_val_score(model, X_train_reg_scaled, y_train_reg, cv=5, scoring='r2')
    
    reg_results[name] = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'cv_mean': cv_scores.mean(),
        'predictions': y_pred_reg
    }
    
    print(f'\n=== نتائج نموذج: {name} (انحدار) ===')
    print(f'📉 متوسط مربعات الخطأ MSE: {mse:.4f}')
    print(f'📏 جذر متوسط مربعات الخطأ RMSE: {rmse:.4f} (درجة معدل)')
    print(f'📊 متوسط الخطأ المطلق MAE: {mae:.4f} (درجة معدل)')
    print(f'📈 معامل التحديد R2 Score: {r2:.2%}')
    print(f'🔄 متوسط التحقق المتقاطع للـ R2: {cv_scores.mean():.2%} (+/- {cv_scores.std()*2:.4f})')

print('\n=== جدول مقارنة نماذج الانحدار الأولية ===')
for name, res in reg_results.items():
    print(f'📌 {name} -> R2 (الدقة): {res["r2"]:.2%}, متوسط الخطأ RMSE: {res["rmse"]:.4f}')

In [24]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

reg_names = list(reg_results.keys())
r2_scores = [reg_results[m]['r2'] for m in reg_names]
rmse_scores = [reg_results[m]['rmse'] for m in reg_names]

x = np.arange(len(reg_names))
width = 0.35

axes[0].bar(x - width/2, r2_scores, width, label='R2 Score (معامل التحديد)', color='skyblue')
axes[0].bar(x + width/2, rmse_scores, width, label='RMSE (جذر نسبة الخطأ)', color='salmon')
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_names, rotation=15)
axes[0].set_ylabel('القيمة (Score)')
axes[0].set_title('مقارنة مقاييس أداء نماذج توقع المعدل')
axes[0].legend()

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['r2'])
y_pred_best_reg = reg_results[best_reg_name]['predictions']

axes[1].scatter(y_test_reg, y_pred_best_reg, alpha=0.5, color='teal')
axes[1].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
axes[1].set_xlabel('المعدل التراكمي الحقيقي للطالب (Actual GPA)')
axes[1].set_ylabel('المعدل التراكمي الذي توقعه النموذج (Predicted GPA)')
axes[1].set_title(f'المعدل الفعلي مقابل التنبؤ لـ {best_reg_name}')

plt.tight_layout()
plt.savefig('data/regression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('تم بنجاح رسم وحفظ مقارنة تنبؤات المعدل الدراسي في مجلد data!')

## Classification Models (Track Prediction)

In [25]:

cls_param_grids = {
    'Decision Tree': {
        'max_depth': [10, 15, 20, 25, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [15, 20, 25, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
}

cls_tuned_results = {}

for name, model in cls_models.items():
    print(f'\n⚙️ جاري تشغيل البحث الشبكي لضبط نموذج: {name}...')
    
    grid_search = GridSearchCV(model, cls_param_grids[name], cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
    
    grid_search.fit(X_train_cls_scaled, y_train_cls)
    
    best_model_tuned = grid_search.best_estimator_
    
    y_pred_tuned = best_model_tuned.predict(X_test_cls_scaled)
    
    acc_tuned = accuracy_score(y_test_cls, y_pred_tuned)
    f1_tuned = f1_score(y_test_cls, y_pred_tuned, average='weighted')
    
    cls_tuned_results[name] = {
        'model': best_model_tuned,
        'accuracy': acc_tuned,
        'f1_score': f1_tuned,
        'best_params': grid_search.best_params_,
        'predictions': y_pred_tuned
    }
    
    print(f'✅ دقة نموذج {name} بعد الضبط: {acc_tuned:.2%}')
    print(f'🏷️ أفضل إعدادات تم اختيارها تلقائياً: {grid_search.best_params_}')

## Regression Models (GPA Prediction)

In [26]:

reg_param_grids = {
    'Linear Regression': {},
    'Random Forest Regressor': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
}

reg_tuned_results = {}

for name, model in reg_models.items():
    if not reg_param_grids[name]:
        print(f'\n=== نموذج: {name} (انحدار) - لا يتطلب ضبط معاملات فائقة ===')
        y_pred_reg = model.predict(X_test_reg_scaled)
        r2 = r2_score(y_test_reg, y_pred_reg)
        rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
        
        reg_tuned_results[name] = {
            'model': model,
            'r2': r2,
            'rmse': rmse,
            'best_params': {},
            'predictions': y_pred_reg
        }
        print(f'📈 معامل التحديد R2: {r2:.2%} | 📏 متوسط الخطأ RMSE: {rmse:.4f}')
        continue
    
    print(f'\n⚙️ جاري ضبط معاملات نموذج: {name} (انحدار)...')
    grid_search = GridSearchCV(model, reg_param_grids[name], cv=5, scoring='r2', n_jobs=-1, verbose=0)
    grid_search.fit(X_train_reg_scaled, y_train_reg)
    
    best_model_tuned = grid_search.best_estimator_
    y_pred_tuned = best_model_tuned.predict(X_test_reg_scaled)
    
    r2_tuned = r2_score(y_test_reg, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test_reg, y_pred_tuned))
    
    reg_tuned_results[name] = {
        'model': best_model_tuned,
        'r2': r2_tuned,
        'rmse': rmse_tuned,
        'best_params': grid_search.best_params_,
        'predictions': y_pred_tuned
    }
    
    print(f'✅ معامل التحديد R2 بعد الضبط: {r2_tuned:.2%}')
    print(f'📏 متوسط الخطأ RMSE بعد الضبط: {rmse_tuned:.4f} درجة معدل')
    print(f'🏷️ أفضل إعدادات تم اختيارها تلقائياً: {grid_search.best_params_}')

## Final Model Comparison

In [27]:

print('\n' + '=' * 80)
print('🏆 جدول المقارنة النهائي لنماذج تصنيف وتنبؤ المسار الأكاديمي (Track Prediction) 🏆')
print('=' * 80)

cls_comparison_df = pd.DataFrame({
    'Model (النموذج)': list(cls_tuned_results.keys()),
    'Accuracy (الدقة العامة)': [cls_tuned_results[m]['accuracy'] for m in cls_tuned_results],
    'F1 Score (مقياس الجودة)': [cls_tuned_results[m]['f1_score'] for m in cls_tuned_results]
}).sort_values('Accuracy (الدقة العامة)', ascending=False)

print(cls_comparison_df.to_string(index=False))

best_cls = max(cls_tuned_results, key=lambda k: cls_tuned_results[k]['accuracy'])
print(f'\n👑 *** أفضل نموذج تصنيف فائز للمشروع: {best_cls} بدقة ممتازة: {cls_tuned_results[best_cls]["accuracy"]:.2%} ***')
print(f'⚙️ الإعدادات الذهبية المثالية له: {cls_tuned_results[best_cls]["best_params"]}')

In [28]:

print('\n' + '=' * 80)
print('🏆 جدول المقارنة النهائي لنماذج تنبؤ المعدل الدراسي (GPA Prediction) 🏆')
print('=' * 80)

reg_comparison_df = pd.DataFrame({
    'Model (النموذج)': list(reg_tuned_results.keys()),
    'R2 Score (معامل التحديد)': [reg_tuned_results[m]['r2'] for m in reg_tuned_results],
    'RMSE (نسبة الخطأ)': [reg_tuned_results[m]['rmse'] for m in reg_tuned_results]
}).sort_values('R2 Score (معامل التحديد)', ascending=False)

print(reg_comparison_df.to_string(index=False))

best_reg = max(reg_tuned_results, key=lambda k: reg_tuned_results[k]['r2'])
print(f'\n👑 *** أفضل نموذج انحدار فائز للمشروع: {best_reg} بمعامل تحديد R2: {reg_tuned_results[best_reg]["r2"]:.2%} ***')
print(f'⚙️ الإعدادات المثالية للنموذج الفائز: {reg_tuned_results[best_reg]["best_params"]}')

In [29]:

print(f'\n=== تقرير الأداء الشامل والتفصيلي لأفضل نموذج تصنيف فائز ({best_cls}) ===')
print(classification_report(
    y_test_cls, 
    cls_tuned_results[best_cls]['predictions'], 
    target_names=le.classes_
))

In [30]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cls_names = list(cls_tuned_results.keys())
acc_tuned = [cls_tuned_results[m]['accuracy'] for m in cls_names]
f1_tuned = [cls_tuned_results[m]['f1_score'] for m in cls_names]

x = np.arange(len(cls_names))
width = 0.35

axes[0].bar(x - width/2, acc_tuned, width, label='Accuracy (الدقة)', color='skyblue')
axes[0].bar(x + width/2, f1_tuned, width, label='F1 Score (مقياس F1)', color='lightgreen')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cls_names, rotation=15)
axes[0].set_ylabel('القيمة (Score)')
axes[0].set_title('مقارنة نماذج التصنيف بعد الضبط')
axes[0].legend()
axes[0].set_ylim(0.85, 1.0)

cm_best = confusion_matrix(y_test_cls, cls_tuned_results[best_cls]['predictions'])
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title(f'مصفوفة الحيرة لـ {best_cls}')
axes[1].set_xlabel('المسار المتوقع بواسطة النموذج (Predicted)')
axes[1].set_ylabel('المسار الفعلي للطالب (Actual)')
axes[1].tick_params(axis='x', rotation=45)

reg_names_tuned = list(reg_tuned_results.keys())
r2_tuned = [reg_tuned_results[m]['r2'] for m in reg_names_tuned]
rmse_tuned = [reg_tuned_results[m]['rmse'] for m in reg_names_tuned]

x2 = np.arange(len(reg_names_tuned))
axes[2].bar(x2 - width/2, r2_tuned, width, label='R2 Score (معامل التحديد)', color='skyblue')
axes[2].bar(x2 + width/2, rmse_tuned, width, label='RMSE (نسبة الخطأ)', color='salmon')
axes[2].set_xticks(x2)
axes[2].set_xticklabels(reg_names_tuned, rotation=15)
axes[2].set_ylabel('القيمة (Score)')
axes[2].set_title('مقارنة نماذج الانحدار بعد الضبط')
axes[2].legend()

plt.tight_layout()
plt.savefig('data/final_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('تم حفظ اللوحة الثلاثية التقييمية ومصفوفة الحيرة بنجاح في مجلد data!')

## Save Models and Artifacts

In [31]:

os.makedirs('models', exist_ok=True)

for name, res in cls_tuned_results.items():
    filename = f'models/{name.lower().replace(" ", "_")}_model.pkl'
    joblib.dump(res['model'], filename)
    print(f'📁 تم حفظ ملف نموذج تصنيف {name} في المجلد models!')

for name, res in reg_tuned_results.items():
    filename = f'models/{name.lower().replace(" ", "_")}_model.pkl'
    joblib.dump(res['model'], filename)
    print(f'📁 تم حفظ ملف نموذج انحدار {name} في المجلد models!')

artifacts = {
    'scaler_cls': scaler_cls,
    'scaler_reg': scaler_reg,
    'label_encoder': le,
    'feature_columns': feature_columns,
    'regression_features': regression_features,
    'all_courses': all_courses,
    'best_cls_model_name': best_cls,
    'best_reg_model_name': best_reg,
    'association_rules': track_rules.head(50).to_dict()
}

joblib.dump(artifacts, 'models/preprocessing_artifacts.pkl')

print(f'\n🎉 تم بنجاح حفظ ملف كائنات التجهيز المسبق (preprocessing_artifacts.pkl) وهو جاهز للتشغيل الفوري!')
print(f'🌟 أفضل نموذج تصنيف فائز: {best_cls} (دقة: {cls_tuned_results[best_cls]["accuracy"]:.2%})')
print(f'🌟 أفضل نموذج انحدار فائز: {best_reg} (معامل R2: {reg_tuned_results[best_reg]["r2"]:.2%})')

## Section 46

In [32]:

def predict_track(gpa, credit_hours, passed_courses, failed_courses, interest_encoded, attendance, study_hours, prev_gpa):
    input_data = {col: 0 for col in feature_columns}
    
    for course in passed_courses:
        if f'Passed_{course}' in input_data:
            input_data[f'Passed_{course}'] = 1
    
    for course in failed_courses:
        if f'Failed_{course}' in input_data:
            input_data[f'Failed_{course}'] = 1
    
    input_data['Total_Passed'] = len(passed_courses)
    input_data['Total_Failed'] = len(failed_courses)
    input_data['Pass_Fail_Ratio'] = len(passed_courses) / (len(failed_courses) + 1)
    input_data['GPA_Credit_Ratio'] = gpa / (credit_hours / 100.0)
    
    input_data['CS_Passed'] = sum(1 for c in passed_courses if c.startswith('CS'))
    input_data['Math_Passed'] = sum(1 for c in passed_courses if c.startswith('MATH'))
    input_data['Eng_Passed'] = sum(1 for c in passed_courses if c.startswith('ENG'))
    input_data['CS_Failed'] = sum(1 for c in failed_courses if c.startswith('CS'))
    input_data['Math_Failed'] = sum(1 for c in failed_courses if c.startswith('MATH'))
    input_data['Eng_Failed'] = sum(1 for c in failed_courses if c.startswith('ENG'))
    
    input_data['GPA'] = gpa
    input_data['Credit_Hours_Completed'] = credit_hours
    input_data['Attendance_Rate'] = attendance
    input_data['Study_Hours_Per_Week'] = study_hours
    input_data['Previous_Semester_GPA'] = prev_gpa
    input_data['Interest_Encoded'] = interest_encoded
    
    input_df = pd.DataFrame([input_data])
    input_scaled = scaler_cls.transform(input_df)
    
    best_model = cls_tuned_results[best_cls]['model']
    prediction = best_model.predict(input_scaled)[0]
    probabilities = best_model.predict_proba(input_scaled)[0]
    
    track_name = le.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    track_probs = list(zip(le.classes_, probabilities))
    track_probs.sort(key=lambda x: x[1], reverse=True)
    
    return track_name, confidence, track_probs[:3]


test_passed = ['CS101', 'CS102', 'CS201', 'CS301', 'CS302', 'CS401', 'MATH101', 'MATH201', 'MATH301', 'MATH302', 'ENG101']
test_failed = ['CS220']

track, conf, top3 = predict_track(
    gpa=3.45,
    credit_hours=100,
    passed_courses=test_passed,
    failed_courses=test_failed,
    interest_encoded=0,
    attendance=88.5,
    study_hours=18.0,
    prev_gpa=3.50
)

print(f'=== نتيجة توقع المسار الدراسي وتوجيه الطالب التجريبي ===')
print(f'🎯 التخصص الأساسي الموصى به للطالب: {track}')
print(f'📊 نسبة ثقة ويقين النموذج في الترشيح: {conf:.2%}')
print(f'\n📋 أفضل 3 تخصصات/مسارات مرشحة للطالب بالترتيب:')
for t, p in top3:
    print(f'   ← {t}: بنسبة ملاءمة تعادل {p:.2%}')

In [33]:

def predict_gpa(credit_hours, passed_courses, failed_courses, interest_encoded, attendance, study_hours, prev_gpa):
    input_data = {col: 0 for col in regression_features}
    
    for course in passed_courses:
        if f'Passed_{course}' in input_data:
            input_data[f'Passed_{course}'] = 1
            
    for course in failed_courses:
        if f'Failed_{course}' in input_data:
            input_data[f'Failed_{course}'] = 1
            
    input_data['Total_Passed'] = len(passed_courses)
    input_data['Total_Failed'] = len(failed_courses)
    input_data['Pass_Fail_Ratio'] = len(passed_courses) / (len(failed_courses) + 1)
    
    input_data['GPA_Credit_Ratio'] = 3.0 / (credit_hours / 100.0)
    
    input_data['CS_Passed'] = sum(1 for c in passed_courses if c.startswith('CS'))
    input_data['Math_Passed'] = sum(1 for c in passed_courses if c.startswith('MATH'))
    input_data['Eng_Passed'] = sum(1 for c in passed_courses if c.startswith('ENG'))
    input_data['CS_Failed'] = sum(1 for c in failed_courses if c.startswith('CS'))
    input_data['Math_Failed'] = sum(1 for c in failed_courses if c.startswith('MATH'))
    input_data['Eng_Failed'] = sum(1 for c in failed_courses if c.startswith('ENG'))
    
    input_data['Credit_Hours_Completed'] = credit_hours
    input_data['Attendance_Rate'] = attendance
    input_data['Study_Hours_Per_Week'] = study_hours
    input_data['Previous_Semester_GPA'] = prev_gpa
    input_data['Interest_Encoded'] = interest_encoded
    
    input_df = pd.DataFrame([input_data])
    input_scaled = scaler_reg.transform(input_df)
    
    best_reg_model = reg_tuned_results[best_reg]['model']
    predicted_gpa = best_reg_model.predict(input_scaled)[0]
    
    predicted_gpa = np.clip(predicted_gpa, 1.5, 4.0)
    
    return predicted_gpa


predicted_gpa = predict_gpa(
    credit_hours=100, 
    passed_courses=test_passed, 
    failed_courses=test_failed,
    interest_encoded=0, 
    attendance=88.5, 
    study_hours=18.0, 
    prev_gpa=3.50
)

print(f'=== نتيجة تنبؤ المعدل الدراسي المتوقع للطالب الافتراضي ===')
print(f'📈 المعدل المتوقع للطالب في الفصل القادم: {predicted_gpa:.2f} من 4.00')

## Feature Ablation Study

In [ ]:

feature_columns_no_ie = [col for col in feature_columns if col != 'Interest_Encoded']
regression_features_no_ie = [col for col in regression_features if col != 'Interest_Encoded']

X_no_ie = df[feature_columns_no_ie]
X_train_no_ie, X_test_no_ie, y_train_no_ie, y_test_no_ie = train_test_split(
    X_no_ie, y_track_encoded,
    test_size=0.2, random_state=42, stratify=y_track_encoded
)
scaler_no_ie = StandardScaler()
X_train_no_ie_scaled = scaler_no_ie.fit_transform(X_train_no_ie)
X_test_no_ie_scaled  = scaler_no_ie.transform(X_test_no_ie)

print(f'عدد الميزات مع Interest_Encoded:    {len(feature_columns)} ميزة')
print(f'عدد الميزات بدون Interest_Encoded: {len(feature_columns_no_ie)} ميزة')
print(f'الميزة المحذوفة: Interest_Encoded فقط')
print(f'حجم التدريب: {X_train_no_ie.shape[0]} | حجم الاختبار: {X_test_no_ie.shape[0]}')

In [ ]:

cls_models_no_ie = {
    'Decision Tree':  DecisionTreeClassifier(random_state=42, max_depth=20,
                      min_samples_split=5, min_samples_leaf=2, criterion='gini'),
    'Random Forest':  RandomForestClassifier(random_state=42, n_estimators=200,
                      max_depth=20, min_samples_split=5, min_samples_leaf=2),
    'KNN':            KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan'),
}

cls_results_no_ie = {}
for name, model in cls_models_no_ie.items():
    model.fit(X_train_no_ie_scaled, y_train_no_ie)
    y_pred = model.predict(X_test_no_ie_scaled)
    acc = accuracy_score(y_test_no_ie, y_pred)
    f1  = f1_score(y_test_no_ie, y_pred, average='weighted')
    cls_results_no_ie[name] = {'accuracy': acc, 'f1_score': f1}
    print(f'{name}: Accuracy={acc:.4f} ({acc:.2%})  |  F1={f1:.4f}')

print('\n✅ اكتمل تدريب النماذج بدون Interest_Encoded!')

In [ ]:

models_list = ['Decision Tree', 'Random Forest', 'KNN']

comparison_data = {
    'النموذج': models_list,
    'Accuracy مع IE':    [cls_tuned_results[m]['accuracy']      for m in models_list],
    'Accuracy بدون IE':  [cls_results_no_ie[m]['accuracy']      for m in models_list],
    'الفرق (%)':         [round((cls_tuned_results[m]['accuracy'] - cls_results_no_ie[m]['accuracy'])*100, 3)
                           for m in models_list],
    'F1 مع IE':          [cls_tuned_results[m]['f1_score']       for m in models_list],
    'F1 بدون IE':        [cls_results_no_ie[m]['f1_score']       for m in models_list],
}

cmp_df = pd.DataFrame(comparison_data)
print('=' * 80)
print('جدول المقارنة الشامل: تأثير ميزة Interest_Encoded على أداء النماذج')
print('=' * 80)
print(cmp_df.to_string(index=False))
print()

for m in models_list:
    diff = cls_tuned_results[m]['accuracy'] - cls_results_no_ie[m]['accuracy']
    direction = 'تحسّن 📈' if diff > 0 else 'تراجع 📉' if diff < 0 else 'لا تغيير ⚖️'
    print(f'• {m}: {direction} بمقدار {abs(diff)*100:.3f}%')

avg_diff = np.mean([(cls_tuned_results[m]['accuracy'] - cls_results_no_ie[m]['accuracy']) for m in models_list])
print(f'\nمتوسط التأثير: {avg_diff*100:+.3f}%')
if abs(avg_diff) > 0.02:
    print('📌 الاستنتاج: Interest_Encoded ميزة ذات تأثير كبير وجوهري على الدقة!')
elif abs(avg_diff) > 0.005:
    print('📌 الاستنتاج: Interest_Encoded ميزة مفيدة ذات تأثير متوسط.')
else:
    print('📌 الاستنتاج: تأثير Interest_Encoded ضئيل. النماذج تعتمد بشكل رئيسي على السجل الأكاديمي.')

In [ ]:

x = np.arange(len(models_list))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(x - width/2,
                     [cls_tuned_results[m]['accuracy'] for m in models_list],
                     width, label='مع Interest_Encoded', color='#6366f1', alpha=0.85)
bars2 = axes[0].bar(x + width/2,
                     [cls_results_no_ie[m]['accuracy'] for m in models_list],
                     width, label='بدون Interest_Encoded', color='#f97316', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_list)
axes[0].set_ylabel('الدقة (Accuracy)')
axes[0].set_title('تأثير Interest_Encoded على دقة التصنيف')
axes[0].legend()
axes[0].set_ylim(0.85, 1.02)
for bar in [*bars1, *bars2]:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+0.001, f'{h:.2%}',
                 ha='center', va='bottom', fontsize=9)

bars3 = axes[1].bar(x - width/2,
                     [cls_tuned_results[m]['f1_score'] for m in models_list],
                     width, label='مع IE', color='#6366f1', alpha=0.85)
bars4 = axes[1].bar(x + width/2,
                     [cls_results_no_ie[m]['f1_score'] for m in models_list],
                     width, label='بدون IE', color='#f97316', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_list)
axes[1].set_ylabel('مقياس F1 Score')
axes[1].set_title('تأثير Interest_Encoded على مقياس F1')
axes[1].legend()
axes[1].set_ylim(0.85, 1.02)
for bar in [*bars3, *bars4]:
    h = bar.get_height()
    axes[1].text(bar.get_x()+bar.get_width()/2, h+0.001, f'{h:.3f}',
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('data/encoding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ تم حفظ رسم المقارنة في data/encoding_comparison.png')

## 3. Exploratory Data Analysis (EDA)